# Análise do Modelo — NAO Pose Estimation

Avalia o checkpoint treinado pelo `scripts/train.py`:
- Metadados do checkpoint (época, PCK salvo, backbone)
- Contagem de parâmetros
- PCK@0.2 recalculado no conjunto de validação
- Acurácia por keypoint
- Visualização de predições (skeleton overlay)
- Heatmaps preditos
- Distribuição de erros

In [8]:
from pathlib import Path

# ── Caminhos (ajuste se necessário) ──────────────────────────────────────────
REPO_ROOT  = Path("../")                         # dataset_generator/
CKPT_PATH  = REPO_ROOT / "checkpoints/best.pt"  # ou last.pt
DATA_ROOT  = REPO_ROOT / "output/output"         # onde estão images/ e annotations/

# Verifica se o DATA_ROOT existe; tenta alternativa plana
if not DATA_ROOT.exists():
    DATA_ROOT = REPO_ROOT / "output"
    print(f"[INFO] Usando DATA_ROOT alternativo: {DATA_ROOT}")

print(f"CKPT_PATH : {CKPT_PATH.resolve()}")
print(f"DATA_ROOT : {DATA_ROOT.resolve()}")
print(f"Checkpoint existe: {CKPT_PATH.exists()}")

[INFO] Usando DATA_ROOT alternativo: ..\output
CKPT_PATH : C:\Users\Rinaldo\Documents\projetos\synthpose-webots\dataset_generator\checkpoints\best.pt
DATA_ROOT : C:\Users\Rinaldo\Documents\projetos\synthpose-webots\dataset_generator\output
Checkpoint existe: True


In [9]:
%pip install opencv-python
%pip install matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\Rinaldo\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\Rinaldo\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [10]:
import json
import math
import numpy as np
import cv2
import torch
import torch.nn as nn
import torchvision.models as models
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


## 1. Definição do Modelo

In [11]:
IMG_H, IMG_W = 256, 192
HM_H,  HM_W  = 64,  48
SIGMA         = 2.0
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

KP_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle",
]

# Skeleton 0-based (convertido do COCO 1-based)
SKELETON = [
    [15,13],[13,11],[16,14],[14,12],[11,12],
    [5,11],[6,12],[5,6],[5,7],[6,8],[7,9],[8,10],
    [1,2],[0,1],[0,2],[1,3],[2,4],[3,5],[4,6],
]


def _deconv_block(in_ch: int, out_ch: int) -> nn.Sequential:
    return nn.Sequential(
        nn.ConvTranspose2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    )


class SimpleBaseline(nn.Module):
    def __init__(self, backbone: str = "resnet50", num_kp: int = 17):
        super().__init__()
        net = getattr(models, backbone)(weights="DEFAULT")
        self.backbone = nn.Sequential(*list(net.children())[:-2])
        in_ch = 2048 if backbone in ("resnet50", "resnet101", "resnet152") else 512
        self.decoder = nn.Sequential(
            _deconv_block(in_ch, 256),
            _deconv_block(256,   256),
            _deconv_block(256,   256),
        )
        self.head = nn.Conv2d(256, num_kp, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.decoder(self.backbone(x)))

## 2. Checkpoint — Metadados e Parâmetros

In [12]:
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)

backbone = ckpt.get("backbone", "resnet50")
epoch    = ckpt.get("epoch", "?")
pck_saved = ckpt.get("pck", float("nan"))

model = SimpleBaseline(backbone=backbone).to(DEVICE)
model.load_state_dict(ckpt["state_dict"])
model.eval()

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Backbone       : {backbone}")
print(f"Época salva    : {epoch}")
print(f"PCK@0.2 (salvo): {pck_saved:.4f}")
print(f"Parâmetros totais    : {total_params:,}")
print(f"Parâmetros treináveis: {trainable_params:,}")

Backbone       : resnet50
Época salva    : 17
PCK@0.2 (salvo): 0.9983
Parâmetros totais    : 33,999,697
Parâmetros treináveis: 33,999,697


## 3. Dataset de Validação

In [ ]:
_FLIP_PAIRS = [(1,2),(3,4),(5,6),(7,8),(9,10),(11,12),(13,14),(15,16)]


def _make_gaussian(cy: float, cx: float, sigma: float = SIGMA) -> np.ndarray:
    hm = np.zeros((HM_H, HM_W), dtype=np.float32)
    x0, y0 = round(cx), round(cy)
    half = int(3 * sigma)
    x1, x2 = max(0, x0 - half), min(HM_W, x0 + half + 1)
    y1, y2 = max(0, y0 - half), min(HM_H, y0 + half + 1)
    if x1 >= x2 or y1 >= y2:
        return hm
    xs = np.arange(x1, x2) - x0
    ys = np.arange(y1, y2) - y0
    xx, yy = np.meshgrid(xs, ys)
    hm[y1:y2, x1:x2] = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    return hm


class NaoPoseDataset(Dataset):
    def __init__(self, img_root: Path, ann_file: Path):
        self.img_root = img_root
        with open(ann_file) as f:
            raw = json.load(f)
        ann_by_id = {a["image_id"]: a for a in raw["annotations"]}
        self.samples = [
            (im, ann_by_id[im["id"]])
            for im in raw["images"]
            if im["id"] in ann_by_id
        ]

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        img_info, ann = self.samples[idx]
        path = self.img_root / img_info["file_name"]
        img  = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
        orig_h, orig_w = img.shape[:2]
        img_resized = cv2.resize(img, (IMG_W, IMG_H)).astype(np.float32) / 255.0
        img_norm = (img_resized - MEAN) / STD
        img_t = torch.from_numpy(img_norm.transpose(2, 0, 1).copy())
        kps = np.array(ann["keypoints"], dtype=np.float32).reshape(17, 3)
        sx, sy = HM_W / orig_w, HM_H / orig_h
        heatmaps = np.zeros((17, HM_H, HM_W), dtype=np.float32)
        weights  = np.zeros(17, dtype=np.float32)
        for k in range(17):
            x, y, v = kps[k]
            if v == 0:
                continue
            heatmaps[k] = _make_gaussian(y * sy, x * sx)
            weights[k]  = 1.0 if v == 2 else 0.5
        return img_t, torch.from_numpy(heatmaps), torch.from_numpy(weights), img, kps


val_ann = DATA_ROOT / "annotations/person_keypoints_val.json"
val_ds  = NaoPoseDataset(DATA_ROOT / "images", val_ann)
print(f"Amostras de validação: {len(val_ds)}")

Amostras de validação: 101


: 

## 4. PCK@0.2 — Conjunto de Validação Completo

In [ ]:
def decode_heatmaps(hm: torch.Tensor) -> torch.Tensor:
    """Argmax → coordenadas (B, 17, 2) em pixels do heatmap."""
    B, K, H, W = hm.shape
    flat = hm.view(B, K, -1)
    idx  = flat.argmax(dim=2)
    return torch.stack([idx % W, idx // W], dim=2).float()


def collate_fn(batch):
    imgs_t  = torch.stack([b[0] for b in batch])
    hms     = torch.stack([b[1] for b in batch])
    wts     = torch.stack([b[2] for b in batch])
    orig_imgs = [b[3] for b in batch]
    kps_list  = [b[4] for b in batch]
    return imgs_t, hms, wts, orig_imgs, kps_list


val_dl = DataLoader(val_ds, batch_size=16, shuffle=False,
                    num_workers=0, collate_fn=collate_fn)

diag = (HM_H**2 + HM_W**2) ** 0.5
per_kp_correct = np.zeros(17)
per_kp_total   = np.zeros(17)
all_errors     = []  # distâncias absolutas (pixels heatmap) de todos os kps visíveis

model.eval()
with torch.no_grad():
    for imgs_t, hms, wts, _, _ in val_dl:
        imgs_t = imgs_t.to(DEVICE)
        pred   = model(imgs_t).cpu()
        pred_coords = decode_heatmaps(pred)      # (B, 17, 2)
        gt_coords   = decode_heatmaps(hms)       # (B, 17, 2)
        dist = (pred_coords - gt_coords).norm(dim=2)  # (B, 17)
        mask = wts > 0                               # (B, 17)
        for k in range(17):
            m = mask[:, k]
            per_kp_correct[k] += ((dist[:, k] / diag < 0.2) & m).float().sum().item()
            per_kp_total[k]   += m.float().sum().item()
            all_errors.extend(dist[:, k][m].numpy().tolist())

per_kp_pck = np.where(per_kp_total > 0, per_kp_correct / per_kp_total, np.nan)
overall_pck = per_kp_correct.sum() / per_kp_total.sum()

print(f"PCK@0.2 geral : {overall_pck:.4f}")
print(f"Comparação com salvo: {pck_saved:.4f}  (diff={overall_pck - pck_saved:+.4f})")

## 5. Acurácia por Keypoint

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
colors = ["#4C72B0" if "left" in n else "#DD8452" if "right" in n else "#55A868"
          for n in KP_NAMES]
bars = ax.bar(KP_NAMES, per_kp_pck, color=colors)
ax.axhline(overall_pck, color="crimson", lw=1.5, ls="--", label=f"Média {overall_pck:.3f}")
ax.set_ylim(0, 1.05)
ax.set_ylabel("PCK@0.2")
ax.set_title("Acurácia por Keypoint (validação)")
ax.set_xticks(range(17))
ax.set_xticklabels(KP_NAMES, rotation=40, ha="right", fontsize=9)
legend_patches = [
    mpatches.Patch(color="#4C72B0", label="esquerdo"),
    mpatches.Patch(color="#DD8452", label="direito"),
    mpatches.Patch(color="#55A868", label="centro"),
]
ax.legend(handles=legend_patches + [plt.Line2D([0],[0], color="crimson", ls="--", label=f"Média {overall_pck:.3f}")])
for bar, v in zip(bars, per_kp_pck):
    if not np.isnan(v):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f"{v:.2f}",
                ha="center", va="bottom", fontsize=7)
plt.tight_layout()
plt.show()

## 6. Distribuição de Erros (pixels do heatmap)

In [ ]:
errors = np.array(all_errors)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(errors, bins=60, color="steelblue", edgecolor="white", linewidth=0.5)
axes[0].axvline(np.median(errors), color="crimson", ls="--", label=f"Mediana {np.median(errors):.2f}px")
axes[0].axvline(np.mean(errors),   color="orange",  ls="--", label=f"Média {np.mean(errors):.2f}px")
axes[0].set_xlabel("Erro (px heatmap)")
axes[0].set_ylabel("Frequência")
axes[0].set_title("Distribuição de erros — todos os keypoints")
axes[0].legend()

axes[1].hist(errors, bins=60, cumulative=True, density=True,
             color="steelblue", edgecolor="white", linewidth=0.5)
thr_px = 0.2 * diag
axes[1].axvline(thr_px, color="crimson", ls="--", label=f"limiar PCK@0.2 ({thr_px:.1f}px)")
axes[1].set_xlabel("Erro (px heatmap)")
axes[1].set_ylabel("Fração acumulada")
axes[1].set_title("CDF de erros")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Mediana do erro : {np.median(errors):.2f} px")
print(f"Média do erro   : {np.mean(errors):.2f} px")
print(f"P95 do erro     : {np.percentile(errors, 95):.2f} px")

## 7. Visualização de Predições (skeleton overlay)

In [ ]:
def draw_skeleton(ax, kps_xy, kps_v, color_kp="lime", color_sk="dodgerblue", scale_x=1.0, scale_y=1.0):
    """Desenha skeleton sobre um eixo matplotlib. kps_xy: (17,2) em pixels da imagem original."""
    for a, b in SKELETON:
        if kps_v[a] > 0 and kps_v[b] > 0:
            ax.plot([kps_xy[a,0]*scale_x, kps_xy[b,0]*scale_x],
                    [kps_xy[a,1]*scale_y, kps_xy[b,1]*scale_y],
                    color=color_sk, lw=1.5, alpha=0.85)
    for k in range(17):
        if kps_v[k] > 0:
            ax.scatter(kps_xy[k,0]*scale_x, kps_xy[k,1]*scale_y,
                       c=color_kp, s=20, zorder=5)


def predict_single(img_bgr_or_rgb, is_rgb=True):
    """Roda inferência em uma imagem (H,W,3) e retorna coords (17,2) em px do heatmap."""
    img = img_bgr_or_rgb if is_rgb else cv2.cvtColor(img_bgr_or_rgb, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(img, (IMG_W, IMG_H)).astype(np.float32) / 255.0
    norm = (resized - MEAN) / STD
    t = torch.from_numpy(norm.transpose(2,0,1)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        hm = model(t)[0].cpu().numpy()  # (17, HM_H, HM_W)
    coords = np.array([
        [np.unravel_index(hm[k].argmax(), hm[k].shape)[1],
         np.unravel_index(hm[k].argmax(), hm[k].shape)[0]]
        for k in range(17)
    ], dtype=np.float32)  # (17, 2) em pixels do heatmap
    return hm, coords


N_SHOW = 6
indices = np.random.choice(len(val_ds), N_SHOW, replace=False)

fig, axes = plt.subplots(2, N_SHOW, figsize=(N_SHOW*3, 6))
fig.suptitle("Predições vs Ground Truth  —  Verde: GT  |  Vermelho: Pred", fontsize=11)

for col, idx in enumerate(indices):
    _, _, _, orig_img, kps_gt = val_ds[idx]
    hm_pred, pred_coords = predict_single(orig_img)

    orig_h, orig_w = orig_img.shape[:2]
    sx = orig_w / HM_W
    sy = orig_h / HM_H

    # Linha de cima: imagem com GT e Pred
    ax = axes[0, col]
    ax.imshow(orig_img)
    draw_skeleton(ax, kps_gt[:, :2], kps_gt[:, 2], color_kp="lime", color_sk="lime")
    draw_skeleton(ax, pred_coords, np.ones(17), color_kp="red", color_sk="red",
                  scale_x=sx, scale_y=sy)
    ax.axis("off")
    ax.set_title(f"#{idx}", fontsize=8)

    # Linha de baixo: heatmap somado
    ax2 = axes[1, col]
    ax2.imshow(orig_img)
    ax2.imshow(cv2.resize(hm_pred.sum(0), (orig_w, orig_h)),
               alpha=0.5, cmap="hot")
    ax2.axis("off")

plt.tight_layout()
plt.show()

## 8. Heatmaps por Keypoint — Uma Amostra

In [ ]:
sample_idx = indices[0]  # mesma amostra da primeira coluna acima
_, _, _, orig_img, kps_gt = val_ds[sample_idx]
hm_pred, pred_coords = predict_single(orig_img)
orig_h, orig_w = orig_img.shape[:2]

fig, axes = plt.subplots(3, 6, figsize=(15, 8))
fig.suptitle(f"Heatmaps preditos — amostra #{sample_idx}", fontsize=11)
axes_flat = axes.flatten()

for k in range(17):
    ax = axes_flat[k]
    ax.imshow(orig_img)
    ax.imshow(cv2.resize(hm_pred[k], (orig_w, orig_h)), alpha=0.6, cmap="hot")
    # ponto GT
    if kps_gt[k, 2] > 0:
        ax.scatter(kps_gt[k, 0], kps_gt[k, 1], c="lime", s=30, zorder=5)
    # ponto pred
    ax.scatter(pred_coords[k,0] * orig_w / HM_W,
               pred_coords[k,1] * orig_h / HM_H,
               c="red", s=30, marker="x", zorder=5)
    ax.set_title(KP_NAMES[k], fontsize=7)
    ax.axis("off")

# Esconde eixo extra (17 kps em grid 3x6 = 18 slots)
axes_flat[17].axis("off")
plt.tight_layout()
plt.show()

## 9. Piores Predições

In [ ]:
per_sample_err = []
for i in range(len(val_ds)):
    _, _, wts_i, orig_img, kps_gt = val_ds[i]
    hm_pred_i, pred_coords_i = predict_single(orig_img)
    orig_h, orig_w = orig_img.shape[:2]
    sx, sy = orig_w / HM_W, orig_h / HM_H
    gt_hm = np.stack([
        [kps_gt[k, 0] / sx, kps_gt[k, 1] / sy] for k in range(17)
    ])  # (17,2) em pixels heatmap
    mask = kps_gt[:, 2] > 0
    if mask.sum() == 0:
        per_sample_err.append(0.0)
        continue
    errs = np.linalg.norm(pred_coords_i - gt_hm, axis=1)
    per_sample_err.append(errs[mask].mean())

worst_idx = np.argsort(per_sample_err)[-6:][::-1]
print("Piores amostras (erro médio em px heatmap):")
for i in worst_idx:
    print(f"  idx={i:4d}  erro={per_sample_err[i]:.2f}px")

fig, axes = plt.subplots(1, 6, figsize=(18, 3))
fig.suptitle("Piores predições  —  Verde: GT  |  Vermelho: Pred")
for col, idx in enumerate(worst_idx):
    _, _, _, orig_img, kps_gt = val_ds[idx]
    hm_pred_i, pred_coords_i = predict_single(orig_img)
    orig_h, orig_w = orig_img.shape[:2]
    sx, sy = orig_w / HM_W, orig_h / HM_H
    ax = axes[col]
    ax.imshow(orig_img)
    draw_skeleton(ax, kps_gt[:, :2], kps_gt[:, 2], color_kp="lime", color_sk="lime")
    draw_skeleton(ax, pred_coords_i, np.ones(17), color_kp="red", color_sk="red",
                  scale_x=sx, scale_y=sy)
    ax.set_title(f"#{idx}  err={per_sample_err[idx]:.1f}px", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()